In [12]:
import logging
from dataclasses import dataclass, asdict
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

def safe_float(val: str) -> float:
    clean_val = val.replace(',', '').replace('\xa0', '').strip()
    if not clean_val or clean_val == "-":
        return 0.0
    try:
        return float(clean_val)
    except ValueError:
        return 0.0

def safe_int(val: str) -> int:
    clean_val = val.replace(',', '').replace('\xa0', '').strip()
    if not clean_val or clean_val == "-":
        return 0
    try:
        return int(clean_val)
    except ValueError:
        return 0

@dataclass(frozen=True)
class StockTicker:
    symbol: str
    ltp: float
    pct_change: float
    high: float
    low: float
    open_price: float
    qty: int
    turnover: float

    @classmethod
    def from_soup_row(cls, row):
        """Parses a static BeautifulSoup tr tag into a structured data object."""
        cells = row.find_all("td")
        if len(cells) < 8:
            raise ValueError("Row data contains incomplete columns")
            
        return cls(
            symbol=cells[0].find("a").text.strip(),
            ltp=safe_float(cells[1].text),
            pct_change=safe_float(cells[2].text),
            high=safe_float(cells[3].text),
            low=safe_float(cells[4].text),
            open_price=safe_float(cells[5].text),
            qty=safe_int(cells[6].text),
            turnover=safe_float(cells[7].text)
        )

class WebDriverSession:
    def __enter__(self):
        options = webdriver.ChromeOptions()
        options.add_argument("--headless=new")
        options.add_argument("--start-maximized")
        options.add_argument("--disable-blink-features=AutomationControlled")
        self.driver = webdriver.Chrome(options=options)
        return self.driver

    def __exit__(self, exc_type, exc_val, exc_tb):
        if hasattr(self, 'driver'):
            self.driver.quit()
            logger.info("WebDriver session closed safely.")

def fetch_live_market_data() -> list[StockTicker]:
    url = "https://merolagani.com/LatestMarket.aspx"
    market_tickers = []

    logger.info(f"Initializing scraping sequence for: {url}")
    
    with WebDriverSession() as driver:
        try:
            driver.get(url)
            
            table_xpath = "//table[contains(@class, 'table-index')]"
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, table_xpath)))
            
            # --- PRODUCTION FIX: Capture raw page source snapshot immediately ---
            raw_html = driver.page_source
            logger.info("DOM captured successfully. Passing content to BeautifulSoup memory pipeline...")
            
        except Exception as global_err:
            logger.error(f"Execution failed during browser extraction: {global_err}")
            raise global_err

    
    soup = BeautifulSoup(raw_html, "html.parser")
   
    rows = soup.select("table.table-index tbody tr")
    
    for row in rows:
        
        if not row.find("td"):
            continue
        try:
            ticker = StockTicker.from_soup_row(row)
            market_tickers.append(ticker)
        except Exception as parsing_err:
            logger.warning(f"Skipped parsing static row element due to structural anomaly: {parsing_err}")
            continue

    return market_tickers

import pandas as pd

if __name__ == "__main__":
    try:
       
        stocks = fetch_live_market_data()
        logger.info(f"Successfully processed {len(stocks)} records without DOM interference.")
        
        if stocks:
          
            df = pd.DataFrame([asdict(s) for s in stocks])
            
            
            df.columns = ['Symbol', 'LTP', '% Change', 'High', 'Low', 'Open', 'Qty', 'Turnover']
            
            
            pd.set_option('display.width', 1000)
            pd.set_option('display.colheader_justify', 'center')
            
            # 5. Print the gorgeous tabular output
            print("\n" + "="*80)
            print("                     LIVE NEPSE MARKET REPORT                     ")
            print("="*80)
            print(df.to_string(index=False))  
            print("="*80)
            
 
            df.to_csv("live_market_report.csv", index=False)
            logger.info("Saved copy of live data to 'live_market_report.csv'")
                
    except Exception as e:
        logger.critical(f"Pipeline failure: {e}")

2026-06-26 18:05:51,366 [INFO] Initializing scraping sequence for: https://merolagani.com/LatestMarket.aspx


2026-06-26 18:05:53,751 [INFO] DOM captured successfully. Passing content to BeautifulSoup memory pipeline...


2026-06-26 18:05:55,903 [INFO] WebDriver session closed safely.


2026-06-26 18:05:56,209 [INFO] Successfully processed 20 records without DOM interference.


2026-06-26 18:05:56,236 [INFO] Saved copy of live data to 'live_market_report.csv'



                     LIVE NEPSE MARKET REPORT                     
  Symbol    LTP   % Change  High    Low   Open    Qty    Turnover 
    SNORL  343.5    15.00   343.5  298.7  298.7     70     22276.0
    TPKHL  603.2    14.98   603.2  540.3  540.3    630    378489.0
     YMHL  603.2    14.98   603.2  540.3  540.3    350    209593.0
NMBD89/90 1363.0    10.81  1363.0 1266.9 1266.9  27958  36695160.5
  NIMBD90 1280.0     7.11  1290.4 1230.8 1230.8  31041  40054063.9
   NBLD87 1130.0     4.91  1144.3 1109.4 1109.4 195244 223416001.2
     CITY  407.9     4.51   413.8  387.0  391.0 287346 115458168.7
NMBD87/88 1135.0     4.49  1144.5 1118.7 1118.7  25405  29075755.0
    JHAPA 1308.0     3.81  1326.0 1225.0 1250.0  13500  17500820.3
     BGWT  598.9     3.26   614.7  563.0  597.0   4420   2597812.0
    KMCDB  821.0   -10.76   892.4  820.6  892.4   2723   2275477.7
   SAPDBL  814.0    -9.45   854.1  795.0  854.1 103495  83273665.7
     HEIP  327.9    -4.93   350.0  327.9  344.0   2045    681